In [36]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from utils import *
import sys
from astropy.io import fits
import cloud_model_torch as cmt 
import matplotlib
# Default matplotlib settings
matplotlib.rcParams['image.interpolation'] = 'nearest'
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
# Add after imports
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [38]:
# ========================================================================================================
def cloud_model(x, params=None, obs_back=None):
    # Recall that x contains also the background intensity
    if params is None:
        params = combine_parameters()
    #return cmt.model_synth(x, params)
    if obs_back is None:
        return cmt.model_synth_2clouds_givenbck(x, params, 0)
    return cmt.model_synth_2clouds_givenbck(x, params, obs_back)

In [4]:
# Here we load the data to be fit, and we also load and make the "background". 
# Let's see how will this work. 

data_file = np.load("/home/milic/data/MiHI_halpha_filament/mihi_all_data.npz", allow_pickle=True) 
data = np.copy(data_file['data'][0])
wav = data_file['wav']
print(data.shape) 
del(data_file) 

# Load reconstruction:
data_file = np.load("/home/milic/data/MiHI_halpha_filament/mihi_rvm_reconstruction_averages_skglm.npz", allow_pickle=True)
rdata = np.copy(data_file["new_fit"][0])
rextra = np.copy(data_file["new_extrafit"][0])
print(rdata.shape)
print(rextra.shape)
del(data_file)

bdata = rdata - rextra
print(bdata.shape)

(146, 160, 550)
(146, 160, 550)
(146, 160, 550)
(146, 160, 550)


In [39]:
# If neeed reshape so that there is an extra first dimension for time:
data = data.reshape((1, *data.shape))
rdata = rdata.reshape((1, *rdata.shape))
rextra = rextra.reshape((1, *rextra.shape))
bdata = bdata.reshape((1, *bdata.shape))

NameError: name 'rdata' is not defined

In [6]:
# Keep memory free for now
del(rdata)
del(rextra)

In [ ]:
# ====================================================================
def optimization(optimizer,niterations,parameters,model, xl, obs, obs_back):
    """
    This function performs the optimization of the parameters of the model.
    """
    # Move data to GPU if not already there:
    if not isinstance(obs, torch.Tensor):
        obs = torch.from_numpy(np.array(obs.astype(np.float32)))
    if obs.device != parameters.device:
        obs = obs.to(device)
    # And then the same for obs_back
    if not isinstance(obs_back, torch.Tensor):
        obs_back = torch.from_numpy(np.array(obs_back.astype(np.float32)))
    if obs_back.device != parameters.device:
        obs_back = obs_back.to(device)

    from tqdm import trange
    t = trange(niterations, leave=True)
    for loop in t:
        
        optimizer.zero_grad()        #reset gradients
        parameters = combine_parameters()
        syn = cloud_model(xl, parameters, obs_back)
        syn = syn.reshape(obs.shape[0], obs.shape[1], -1)
        chi2loss = torch.mean(torch.sum((obs - syn)**2.0))
        
        reguloss = chi2loss*0.0
        # reguloss += 1e1*regu2(parameters, 0,img)
        # reguloss += 1e1*regu2(parameters, 1,img) 
        # reguloss += 1e-1*regu2(parameters, 2,img)
        # reguloss += 5e-2*regu2(parameters, 3,img)
        # reguloss += 5e-2*regu2(parameters, 4,img)
        #print(img.device, parameters.device)
        #reguloss += 5e1*regu2(parameters, 6,img)

        # First regularization is that S cannot be negative:
        #print(parameters.shape)
        #print (obs.shape)
        reguloss += 1e6*regu_min(parameters, 0, obs.reshape(1,obs.shape[0],obs.shape[1],-1), 0.0)
        reguloss += 1e6*regu_min(parameters, 5, obs.reshape(1,obs.shape[0],obs.shape[1],-1), 0.0)
        reguloss += 1e6*regu_min(parameters, 1, obs.reshape(1,obs.shape[0],obs.shape[1],-1), 0.0)
        reguloss += 1e6*regu_min(parameters, 6, obs.reshape(1,obs.shape[0],obs.shape[1],-1), 0.0)
        
        

        loss = chi2loss + reguloss

        loss.backward()              #calculate gradients
        optimizer.step()             #step fordward

        if (chi2loss.item() != chi2loss.item()):  #check for NaN
            print("NaN detected. Stopping optimization.")
            
            # Maybe we should look for a specific NaN and see why it happens:
            nan_params = parameters[torch.isnan(parameters)]
            if len(nan_params) > 0:
                print("NaN values found in parameters:")
                print(nan_params)

            # Find specific nans in chisquared then and show the appropriate parameters:
            nan_syn = torch.isnan(syn)
            print ("shape of the nan_syn:", nan_syn.shape)
            if len(nan_syn):
                print("NaN values found in synthetic data:")
                print(syn[nan_syn])
                # Show the corresponding parameters
                print("Corresponding parameters:")
                print(parameters[nan_syn[:,:,0].flatten()])
            break

        t.set_postfix({'loss': loss.item(), 'chi2loss': chi2loss.item(), 'reguloss': reguloss.item()})

    # Return final parameters and synthetic image
    final_params = combine_parameters()
    final_params = final_params.reshape(obs.shape[0], obs.shape[1], -1)
    final_params = final_params.detach().cpu().numpy()
    syn = syn.detach().cpu().numpy()

    # Final cleanup
    torch.cuda.empty_cache()
    
    return final_params, syn


In [30]:
# ================================================================================================================
# Readind the data:

#data_filename = "/mn/stornext/d20/RoCS/carlosjd/colabs/ivan/MIHI/cloudinv_mihi/data/mihi_data.fits"
# data_filename = "/home/milic/data/scratch/mihi_data.fits"
#data_filename = "/home/milic/data/scratch/img_fit.210800..211599.00.00.lr0250.cube.fits"
# Now it is loaded already up there

tofit = data[0,15:-15, 15:-15,100:500]
Iqs = np.mean(tofit[:,:,-20:])
print("info::normalizing the input data to the value: ", Iqs)
tofit /= Iqs
tofit_background = bdata[0,15:-15, 15:-15,100:500]
tofit_background /= Iqs
print("info::the input observations have the shape:", tofit.shape)
print("info::the background of inferred has the shape:", tofit_background.shape)
xl = np.copy(wav[100:500])
ll0 = np.asarray([np.mean(xl)])
xl = [xl,ll0]



# Transform data into pytorch object:
tofit_pt = torch.from_numpy(np.array(tofit.astype(np.float32))).to(device)
# And the same for the background
tofit_background_pt = torch.from_numpy(np.array(tofit_background.astype(np.float32))).to(device)


info::normalizing the input data to the value:  1.0
info::the input observations have the shape: (116, 130, 400)
info::the background of inferred has the shape: (116, 130, 400)


In [31]:
def make_param(init_value, shape, grad=True, device='cpu'):
    param = torch.tensor(init_value, dtype=torch.float32, device=device).repeat(*shape)
    param = nn.Parameter(param, requires_grad=grad)
    return param

In [32]:
# Define independent parameter groups
n_pixels = tofit.shape[0] * tofit.shape[1]
device = tofit_pt.device


# Cloud 1 parameters (each independent)
cloud1_S = make_param(0.4, (n_pixels,), grad=True, device=device)
cloud1_tau = make_param(4.5, (n_pixels,), grad=True, device=device)
cloud1_vlos = make_param(-2.0, (n_pixels,), grad=True, device=device)
cloud1_doppler = make_param(6.0, (n_pixels,), grad=True, device=device)
cloud1_loga = make_param(-4.0, (n_pixels,), grad=False, device=device)

# Cloud 2 parameters (each independent)
cloud2_S = make_param(0.1, (n_pixels,), grad=True, device=device)
cloud2_tau = make_param(1.0, (n_pixels,), grad=True, device=device)
cloud2_vlos = make_param(10.0, (n_pixels,), grad=True, device=device)
cloud2_doppler = make_param(5.0, (n_pixels,), grad=True, device=device)
cloud2_loga = make_param(-5.0, (n_pixels,), grad=False, device=device)

# Function to combine parameters for the model
def combine_parameters():
    return torch.stack([
        cloud1_S, cloud1_tau, cloud1_vlos, cloud1_doppler, cloud1_loga,
        cloud2_S, cloud2_tau, cloud2_vlos, cloud2_doppler, cloud2_loga
    ], dim=1)

# Collect all parameters that require gradients for optimizer
params_to_optimize = []
for param in [cloud1_S, cloud1_tau, cloud1_vlos, cloud1_doppler, cloud1_loga,
              cloud2_S, cloud2_tau, cloud2_vlos, cloud2_doppler, cloud2_loga]:
    if param.requires_grad:
        params_to_optimize.append(param)



In [33]:
# Parameters:
print("info:: parameters:", torch.mean(combine_parameters(), axis=0))

info:: parameters: tensor([ 0.4000,  4.5000, -2.0000,  6.0000, -4.0000,  0.1000,  1.0000, 10.0000,
         5.0000, -5.0000], grad_fn=<MeanBackward1>)


In [34]:
torch.cuda.empty_cache()

In [35]:
# ====================================================================
# Send all the info to the optimization module inside the utils.py file:
optimizer = torch.optim.Adam(params_to_optimize, lr=1e-1)
niter = 1000
mymodel = cloud_model # Model from the above
parameters, fit_spectra = optimization(optimizer=optimizer,niterations=niter,
                                        parameters=combine_parameters(),model=mymodel,xl=xl,obs=tofit_pt, obs_back=tofit_background_pt.reshape(n_pixels,-1))

chisq = np.sum((tofit - fit_spectra)**2.0, axis=2) / 400.0
print(np.mean(chisq[0,0]))


  0%|          | 0/1000 [00:00<?, ?it/s]

 14%|█▍        | 143/1000 [05:10<31:00,  2.17s/it, loss=3.02e+4, chi2loss=1.31e+4, reguloss=1.71e+4]

  0%|          | 0/1000 [00:00<?, ?it/s]

 14%|█▍        | 143/1000 [05:10<31:00,  2.17s/it, loss=3.02e+4, chi2loss=1.31e+4, reguloss=1.71e+4]

NaN detected. Stopping optimization.
shape of the nan_syn: torch.Size([116, 130, 400])
NaN values found in synthetic data:
tensor([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
       grad_fn=<IndexBackward0>)
Corresponding parameters:
tensor([[ 1.0249e+00, -8.8770e-01,  7.4095e+00,  1.9977e-04, -4.0000e+00,
          3.1748e-01,  7.2489e+00,  1.8558e+0

In [ ]:
# Average the parameters for only print:
print("info::average parameters:", np.mean(parameters, axis=(0,1)))

In [ ]:
# Extract and compare 3 random pixels from the fit and the data:
np.random.seed(71)
idxs = np.random.randint(0, tofit.shape[0]*tofit.shape[1], size=3)
plt.figure(figsize=[12,6])
for j, idx in enumerate(idxs):
    plt.subplot(2, 3, j+1)
    plt.plot(tofit.reshape(-1, tofit_pt.shape[2])[idx], label='Data')
    plt.plot(fit_spectra.reshape(-1, fit_spectra.shape[2])[idx], label='Fit')
    plt.title(f"Pixel {idx}")
    plt.xlabel('Wavelength index')
    plt.legend()
    if j == 0:
        plt.ylabel('Intensity')
        
# Vertical lines at:
idxs_lambda = [int(0.1*tofit.shape[2]), int(0.5*tofit.shape[2]), int(0.9*tofit.shape[2])]
for idx_lambda in idxs_lambda:
    plt.axvline(x=idx_lambda, color='r', linestyle='-', label=f'Wavelength {idx_lambda}', alpha=0.5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=[10, 6], sharex=True, sharey=True)
for j, idx in enumerate(idxs_lambda):
    ax_obs = axes[0, j]
    im_obs = ax_obs.imshow(tofit[:, :, idx].T, origin='lower', cmap='gray')
    ax_obs.set_title(f"Obs λ idx {idx}")
    fig.colorbar(im_obs, ax=ax_obs)
    ax_obs.scatter(idxs[j] % tofit.shape[1], idxs[j] // tofit.shape[1], color='red', marker='x', s=100)

    ax_fit = axes[1, j]
    im_fit = ax_fit.imshow(fit_spectra[:, :, idx].T, origin='lower', cmap='gray',
                          vmin=tofit_pt[:, :, idx].min(), vmax=tofit_pt[:, :, idx].max())
    ax_fit.set_title(f"Fit λ idx {idx}")
    fig.colorbar(im_fit, ax=ax_fit)
plt.tight_layout()


In [ ]:
im = plt.imshow(np.log10(chisq).T, origin='lower', cmap='magma')
add_colorbar(im, label=r'$\log_{10}(\chi^2)$')

In [ ]:
labels = ['S cloud', 'tau cloud', 'los velocity cloud', 'doppler width cloud',
          'S cloud2', 'tau cloud2', 'los velocity cloud2',
          'doppler width cloud2']

to_plot = np.array([0,1,2,3,5,6,7,8])

cmaps = ['viridis', 'viridis', 'bwr', 'viridis', 'viridis', 'viridis', 'bwr', 'viridis']

fig, axes = plt.subplots(2, 4, figsize=[16, 8], sharex=True, sharey=True)
ax = axes.flatten()
for i in range(8):
    im = ax[i].imshow(parameters[:, :, to_plot[i]].T, origin='lower', cmap=cmaps[i])
    ax[i].set_title(labels[i])
    fig.colorbar(im, ax=ax[i])
plt.tight_layout()
